In [2]:
!pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 52.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [yfinance]/11 [yfinance]]t-py]

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo để output gọn gàng hơn

# 1. Khai báo rổ 40 cổ phiếu Việt Nam (Sàn HOSE)
# Bao gồm rổ VN30 và 10 mã thanh khoản cao (Ngành thép, chứng khoán, bất động sản, hóa chất...)
vn_symbols = [
    # VN30
    'ACB', 'BCM', 'BID', 'BVH', 'CTG', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 
    'MBB', 'MSN', 'MWG', 'PLX', 'POW', 'SAB', 'SHB', 'SSB', 'SSI', 'STB', 
    'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE',
    # 10 mã Mid/Large cap nổi bật
    'DGC', 'VND', 'KBC', 'DIG', 'HSG', 'NKG', 'VCG', 'KDH', 'GEX', 'HCM'
]

# Thêm hậu tố '.HM' để yfinance hiểu đây là mã trên sàn HOSE (TP.HCM)
tickers = [symbol + '.HM' for symbol in vn_symbols]

# Thiết lập thời gian backtest (Ví dụ: 1 năm gần nhất)
start_date = '2020-04-01'
end_date = '2024-04-01'

# Hàm tính toán Max Drawdown
def calculate_max_drawdown(price_series):
    rolling_max = price_series.cummax()
    drawdown = price_series / rolling_max - 1.0
    max_drawdown = drawdown.min()
    return max_drawdown

def filter_top_stocks(tickers, start, end, top_n=6):
    results = []
    print("Đang tải dữ liệu và tính toán. Vui lòng đợi...\n")
    
    # 2. Tải dữ liệu và tính toán các metrics
    for ticker in tickers:
        try:
            data = yf.download(ticker, start=start, end=end, progress=False)
            if len(data) < 100: # Bỏ qua các mã mới lên sàn chưa đủ dữ liệu
                continue
                
            close_prices = data['Adj Close']
            
            # Tính toán Đà tăng trưởng (Lợi nhuận gộp trong kỳ)
            total_return = (close_prices.iloc[-1] / close_prices.iloc[0]) - 1
            
            # Tính toán Max Drawdown
            max_dd = calculate_max_drawdown(close_prices)
            
            # Loại bỏ đuôi .HM để hiển thị cho đẹp
            clean_ticker = ticker.replace('.HM', '')
            
            results.append({
                'Mã CP': clean_ticker,
                'Lợi nhuận (%)': round(total_return * 100, 2),
                'Max Drawdown (%)': round(max_dd * 100, 2)
            })
        except Exception as e:
            pass # Bỏ qua nếu có lỗi tải dữ liệu

    df = pd.DataFrame(results)
    
    # 3. Xây dựng hệ thống chấm điểm (Scoring Model)
    # Xếp hạng Lợi nhuận (Càng cao càng tốt -> hạng 1 là cao nhất)
    df['Hạng Lợi nhuận'] = df['Lợi nhuận (%)'].rank(ascending=False)
    
    # Xếp hạng Max Drawdown (Càng ít âm càng tốt -> hạng 1 là rủi ro thấp nhất)
    df['Hạng Drawdown'] = df['Max Drawdown (%)'].rank(ascending=False)
    
    # Tính Tổng điểm (Score) = Trung bình cộng của 2 thứ hạng (Điểm càng thấp càng tốt)
    # Có thể tùy chỉnh trọng số ở đây. Ví dụ: Ưu tiên an toàn thì nhân hệ số cho Hạng Drawdown
    df['Tổng Điểm'] = df['Hạng Lợi nhuận'] + df['Hạng Drawdown']
    
    # 4. Sắp xếp và chọn ra top 6 cổ phiếu
    top_stocks = df.sort_values('Tổng Điểm', ascending=True).head(top_n)
    
    # Đánh lại số thứ tự từ 1 đến 6
    top_stocks.index = np.arange(1, len(top_stocks) + 1)
    
    return top_stocks

# Chạy mô hình
top_6_portfolio = filter_top_stocks(tickers, start_date, end_date, top_n=6)

print("=== DANH SÁCH 6 CỔ PHIẾU TỐI ƯU NHẤT TỪ RỔ 40 MÃ ===")
print("(Tiêu chí: Cân bằng giữa Đà tăng trưởng cao và Mức sụt giảm tối đa thấp)\n")
print(top_6_portfolio[['Mã CP', 'Lợi nhuận (%)', 'Max Drawdown (%)', 'Tổng Điểm']])

Đang tải dữ liệu và tính toán. Vui lòng đợi...



HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ACB.HM"}}}
$ACB.HM: possibly delisted; no timezone found

1 Failed download:
['ACB.HM']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BCM.HM"}}}
$BCM.HM: possibly delisted; no timezone found

1 Failed download:
['BCM.HM']: possibly delisted; no timezone found
$BID.HM: possibly delisted; no timezone found

1 Failed download:
['BID.HM']: possibly delisted; no timezone found
$BVH.HM: possibly delisted; no price data found  (1d 2020-04-01 -> 2024-04-01)

1 Failed download:
['BVH.HM']: possibly delisted; no price data found  (1d 2020-04-01 -> 2024-04-01)
$CTG.HM: possibly delisted; no timezone found

1 Failed download:
['CTG.HM']: possibly delisted; no timezone found
$FPT.HM: possibly delisted; no timezone found

1 Failed download:
['FPT.HM']: possibly delisted; no t

KeyError: 'Lợi nhuận (%)'